> ## SOLUTION / ANSWER KEY &mdash; Lab 7.3
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-03-structured-output.ipynb`](../lab-03-structured-output.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.3 &mdash; Structured Output, Not Prose

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Define a SCHEMA: each field's declared type and default
- Coerce a messy record -- fill missing fields, fix wrong types
- Validate that the required fields ended up present

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, and the pipeline logic &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Structured output, not prose](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-07-03"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Structured output is only useful if it's **well-shaped** &mdash; every field present, every value the
right type (deck slide 7). Downstream code (route, draft, validate) indexes `rec["order_id"]` and
`rec["intent"]`, so a record that's **missing a field** or carries a **wrong-typed value** breaks the
pipeline. The fix is a **schema**: declare each field's **type** and **default**, then **coerce** any
raw record into that shape &mdash; fill missing fields from their defaults, and convert wrong types (a
numeric `order_id` becomes the **string** the orders DB is keyed by). Prose for humans, a
**validated schema** for machines.

In [ ]:
raw = {"order_id": 4471, "urgency": "high"}   # order_id is an int; intent & attempts are missing
print("raw (messy, from an upstream extract):", raw)
print("we want a COMPLETE, correctly-typed record out.")

## Your Turn
Complete `coerce` (fill defaults for missing fields; coerce wrong-typed values) and `is_valid`
(the required fields must end up present and non-None).

In [ ]:
SCHEMA = {
    "order_id": {"type": str, "default": None},
    "intent":   {"type": str, "default": "other"},
    "urgency":  {"type": str, "default": "low"},
    "attempts": {"type": int, "default": 0},
}
REQUIRED = ("order_id", "intent")

def coerce(raw):
    out = {}
    for field, spec in SCHEMA.items():
        typ, default = spec["type"], spec["default"]
        if field not in raw:
            out[field] = default
            continue
        val = raw[field]
        if not isinstance(val, typ):
            try:
                val = typ(val)
            except (ValueError, TypeError):
                val = default
        out[field] = val
    return out

def is_valid(rec):
    # required fields must be present AND non-None after coercion
    return all(rec.get(f) is not None for f in REQUIRED)

try:
    print("coerced   :", coerce({"order_id": 4471, "urgency": "high"}))
    print("defaults  :", coerce({"order_id": "5090"}))
    print("valid?    :", is_valid(coerce({"order_id": "4471", "intent": "refund"})))
    print("missing id:", is_valid(coerce({"intent": "refund"})))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a missing field gets its schema default", lambda: coerce({"order_id": "4471"})["urgency"] == "low")
expect_true("a missing typed field gets its typed default (0)", lambda: coerce({"order_id": "4471"})["attempts"] == 0)
expect_true("a wrong-typed order_id is coerced to a STRING", lambda: coerce({"order_id": 4471})["order_id"] == "4471")
expect_true("the coerced order_id is really a str", lambda: isinstance(coerce({"order_id": 4471})["order_id"], str))
expect_true("a numeric string coerces to int for a typed field", lambda: coerce({"attempts": "3"})["attempts"] == 3)
expect_true("a full record with the required fields is valid", lambda: is_valid(coerce({"order_id": "4471", "intent": "refund"})) is True)
expect_true("a record still missing a required field is invalid", lambda: is_valid(coerce({"intent": "refund"})) is False)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
A schema of typed fields with defaults turns a messy, half-filled record into something the pipeline can trust. Coerce first, validate second -- next we produce records by extraction.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>